# Read-only preview
Create a private PDF of proposed replacements, blocked findings and scan scope. No database changes or row-level plan.

Start with [VM setup](docs/QUICKSTART.md). Use a read-only account, run cells in order, and keep output outside this project.

In [ ]:
from pathlib import Path
from datetime import date
from getpass import getpass
from uuid import uuid4
from remediation.runner import Config, connect, discover_event_scope
from remediation.preview_report import preview_only_report
RUN_DATABASE = False
conn = None
print("Preview only; no connection made.")

## 1. Set scope and connect
Use the approved schemas and exact CDM version. Keep TLS verification on. Discovery lists eligible tables and exclusions; check the scope before scanning.

In [ ]:
cfg = Config(
    cdm_schema="omopcdm",
    vocabulary_schema="vocabulary",
    tables=("condition_occurrence",),  # Bootstrap only; discovery below selects all visible event tables.
    cdm_version="5.4",  # Set 5.4 or 5.5 to match CDM_SOURCE.cdm_version.
    as_of=date.today().isoformat(),
)
private_output = Path.home() / "private_remediation_results"  # Change to site-approved storage.
if RUN_DATABASE:
    conn = connect(host=input("Database host: "), port=int(input("Port [5432]: ") or "5432"),
                   dbname=input("Database name: "), user=input("Database user: "),
                   password=getpass("Password (not saved): "), sslmode="verify-full",
                   sslrootcert=input("CA certificate path (blank for system defaults): ") or None)
    cfg, coverage = discover_event_scope(conn, cfg)
    print("Selected available event tables:", cfg.tables)
    for entry in coverage:
        print(entry["table"], entry["column"], entry["state"], entry["policy"])
else:
    print("Connection skipped. Set RUN_DATABASE only in your approved environment.")

## 2. Scan and read the PDF
Look for **PREVIEW - NO CHANGES** and **0 committed changes**. Proposed mappings need review. Missing fields are not zero findings. Keep exact counts and local labels private.

In [ ]:
if RUN_DATABASE and conn is not None:
    report_path = private_output / f"preview_{uuid4()}.pdf"
    result = preview_only_report(conn, cfg, report_path)
    print(result)
else:
    print("Scan skipped; no changes.")

## 3. Close the connection
Run this cell even if the scan fails. For a new report, rerun preview, not apply.

In [ ]:
if conn is not None:
    conn.close()
    conn = None
    print("Connection closed.")